# Import you data
## Read raw data from file
Your raw data form the basis of all analyses. Keep the original dataset unchanged!


In [48]:
df = read.csv("ihsm26_StudentLifeData.isle.csv", header = TRUE, stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
# check
nrow(df)

[1] 457

## Clean data

Use code to document and apply all data cleaning and transformations. This makes your work transparent and reproducible, even years after you have completed your analysis. 

Extend this code with even more cleaning steps if required. Carefully comment your code to document your decisions. 

In [49]:
# replace all "" and "N/A" with NA to correctly define missing values 
df[df == "" | df == "N/A"] = NA
# check
sum(df == "" | df == "N/A", na.rm=T) # should be zero now (no "" left in df)

[1] 0

In [50]:
# replace long value labels
df$SQ7[df$SQ7 == "STEM (Science. Technology. Engineering and Mathematics)"] = "STEM"
table(df$SQ7)


      Arts and Architecture      Economics and Business     Education and Languages      Environmental Sciences Health Science and Medicine 
                         17                          97                          21                          14                         113 
                 Humanities                         Law                       Other                  Psychology             Social Sciences 
                         14                          15                          27                          18                          39 
                       STEM 
                         58 

In [51]:
# Initialize the cleaned variable; unclassifiable values remain NA
df$SQ3_4 = NA

# Austria
df$SQ3_4[df$SQ3 %in% c(
  "at", "AT", "austria", "Austria", "Austria Tirol",
  "Austria/Österreich", "Innsbruck", "österreich",
  "Österreich", "Salzburg", "Styria"
)] = "Austria"

# Germany
df$SQ3_4[df$SQ3 %in% c(
  "Deutschland", "Dresden", "German", "germany", "Germany"
)] = "Germany"

# EU/EEA countries other than Austria and Germany
df$SQ3_4[df$SQ3 %in% c(
  "belgium", "Belgium", "Bratislava", "Czechia", "Denmark",
  "finland", "Finland", "Finland erasmus", "France", "Genoa",
  "Hungary", "italia", "Italy", "lapland", "Magyarország", "Nederland",
  "Netherlands", "Netherlands/ Finland (exchange)",
  "Norway", "Spain", "Sweden", "The Netherlands"
)] = "EU/EEA"

# Countries outside the EU/EEA
df$SQ3_4[df$SQ3 %in% c(
  "America", "Australia", "Bangladesh", "Egypt", "england",
  "India", "Japan", "Mexico", "Mozambique", "Nepal",
  "Scotland", "South Korea", "Swistzerland", "Switzerland",
  "UK", "United Kingdom", "UNITED KINGDOM!!!!!!!",
  "united states", "United States", "US", "USA"
)] = "Other"

# Unclassifiable responses remain NA:
# "Political Science", "The world", "Wie"

# Display frequencies, including missing values
table(df$SQ3_4, useNA = "ifany")


Austria  EU/EEA Germany   Other    <NA> 
    242      71      88      32      24 

## Define the final subsample

In [52]:
# all respondents
table(df$SQ4)


    Bachelor’s degree Doctoral degree (PhD)       Master’s degree                 Other   Secondary education 
                  220                     8                   136                    18                    60 

In [56]:
# exclude respondents who do not match the target population
df = df[df$SQ4 != "Secondary education" & !is.na(df$SQ4),]
# check
table(df$SQ4, useNA = "ifany")


    Bachelor’s degree Doctoral degree (PhD)       Master’s degree                 Other 
                  220                     8                   136                    18 

In [57]:
# Size of final sample
nrow(df)

[1] 382

# Analysis templates

In order to create generic templates, the following sample dataframe will be defined:

In [58]:
# NOTE: This step is NOT required! The new sample dataframe "sdf" is defined for demonstration purposes only. 
# It contains two nominal scales (N1, N2), two ordinal scales (O1, O2), and a single metric scale (M1). 
# In your homework, simply refer to df (as imported from file in the very first cell above) and its respective variables 
sdf = data.frame(N1 = df$SQ3_4, N2 = df$SQ8, O1 = df$R3, O2 = df$DH3, M1 = df$SQ1)
variable_labels = c(N1 = "What is the country of your studies?",
                    N2 = "Are/were you working for pay to make your daily living while studying?",
                    O1 = "How would you rate your sleep quality overall?",
                    O2 = "How much do you trust digital health tools to provide reliable health information?",
                    M1 = "How old are you? (years)"
                  )
summary(sdf)


         N1              N2              O1              O2            M1       
 Length   :382   Length   :382   Length   :382   Length   :382   Min.   :17.00  
 N.unique :  4   N.unique :  2   N.unique :  5   N.unique :  5   1st Qu.:22.00  
 N.blank  :  0   N.blank  :  0   N.blank  :  0   N.blank  :  0   Median :24.00  
 Min.nchar:  5   Min.nchar:  2   Min.nchar:  3   Min.nchar:  8   Mean   :24.36  
 Max.nchar:  7   Max.nchar:  3   Max.nchar:  9   Max.nchar: 10   3rd Qu.:26.00  
 NAs      : 15   NAs      : 39   NAs      :  2   NAs      : 35   Max.   :38.00  
                                                                 NAs    :2      

## Describe distributions

### Nominal Scales

In [62]:
# nominal scales
t = table(sdf$N1, useNA="always")
t = data.frame(Value = names(t), Count = as.numeric(t), Percent = as.numeric(prop.table(t)))
t[c(1,3,2,4,5),]  # sort output as ranking if required

    Value Count    Percent
1 Austria   211 0.55235602
3 Germany    70 0.18324607
2  EU/EEA    58 0.15183246
4   Other    28 0.07329843
5    <NA>    15 0.03926702

### Ordinal Scales

In [63]:
# ordinal scales
t = table(sdf$O1, useNA="always")
t = data.frame(Value = names(t), Count = as.numeric(t), Percent = as.numeric(prop.table(t)))
t[c(5,2,3,1,4,6),] # adjust output to predefined order


      Value Count     Percent
5 very good    42 0.109947644
2      good   150 0.392670157
3   neutral   134 0.350785340
1       bad    49 0.128272251
4  very bad     5 0.013089005
6      <NA>     2 0.005235602

### Metric Scales

In [64]:
# use summary() to get the most relevant descriptive statistics in a single batch, ...
summary(sdf$M1)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
  17.00   22.00   24.00   24.36   26.00   38.00       2 

In [65]:
# ... extend the summary by additional parameters and reorder them to your taste, ...
c(summary(sdf$M1), SD = sd(sdf$M1, na.rm=T))[c(1, 2, 3, 5, 6, 4, 8, 7)]

     Min.   1st Qu.    Median   3rd Qu.      Max.      Mean        SD       NAs 
17.000000 22.000000 24.000000 26.000000 38.000000 24.357895  3.208385  2.000000 

In [66]:
# ... or compile and format your individual selection 
round(c(quantile(sdf$M1, c(.33, .5, .67), na.rm=T), 
s = sd(sdf$M1, na.rm=T),
"s^2" = var(sdf$M1, na.rm=T)), 3)

   33%    50%    67%      s    s^2 
23.000 24.000 25.000  3.208 10.294 

## Crosstabulation (D1)

In [67]:
table(sdf$N1, sdf$N2)

         
           No Yes
  Austria  61 132
  EU/EEA   19  33
  Germany  16  50
  Other     8  15

In [68]:
# add totals
t = table(sdf$N1, sdf$N2)
t = cbind(t, Total = rowSums(t))
t = rbind(t, Total = colSums(t))
t

         No Yes Total
Austria  61 132   193
EU/EEA   19  33    52
Germany  16  50    66
Other     8  15    23
Total   104 230   334

### Total percentages 


In [69]:
# total percentages 
t = table(sdf$N1, sdf$N2)
t = prop.table(t)
t = cbind(t, Total = rowSums(t))
t = rbind(t, Total = colSums(t))
t*100

               No       Yes      Total
Austria 18.263473 39.520958  57.784431
EU/EEA   5.688623  9.880240  15.568862
Germany  4.790419 14.970060  19.760479
Other    2.395210  4.491018   6.886228
Total   31.137725 68.862275 100.000000

### Row percentages 


In [ ]:
# row percentages 
# how does N2 differ between groups of N1?
t = table(sdf$N1, sdf$N2)
t = rbind(t, Total = colSums(t))
# save the total counts for later use
n = rowSums(t)
t = prop.table(t, 1)
t = cbind(t, Total = rowSums(t))
# format as percentages
t = round(t*100, 1)
# we have to add the total counts for each 100% total to allow for evaluating the percentages correctly
t = cbind(t, N = n)
t

          No  Yes Total   N
Austria 31.6 68.4   100 193
EU/EEA  36.5 63.5   100  52
Germany 24.2 75.8   100  66
Other   34.8 65.2   100  23
Total   31.1 68.9   100 334

### Column percentages 


In [ ]:
# column percentages 
# how does N1 differ between groups of N2?
t = table(sdf$N1, sdf$N2)
t = cbind(t, Total = rowSums(t))
# save the total counts for later use
n = colSums(t)
t = prop.table(t, 2)
t = rbind(t, Total = colSums(t))
# format as percentages
t = round(t*100, 1)
# we have to add the total counts for each 100% total to allow for evaluating the percentages correctly
t = rbind(t, N = n)
t

           No   Yes Total
Austria  58.7  57.4  57.8
EU/EEA   18.3  14.3  15.6
Germany  15.4  21.7  19.8
Other     7.7   6.5   6.9
Total   100.0 100.0 100.0
N       104.0 230.0 334.0

## Group parameters (D2)


Use the functions `by()` or `tapply()` to provide grouped parametric statistics. In the following, `tapply()` is used because it provides simpler output. You may replace it with `by()` if you prefer the latter.

In [78]:
# tapply(data, group, statistics)
tapply(sdf$M1, sdf$N1, mean)

 Austria   EU/EEA  Germany    Other 
24.46919       NA 23.90000 26.32143 

In [79]:
# As in the univariate case, you must add the na.rm flag in order to ignore NAs. Simply append any additional input.
tapply(sdf$M1, sdf$N1, mean, na.rm=T)

 Austria   EU/EEA  Germany    Other 
24.46919 23.56140 23.90000 26.32143 

In [90]:
# Note: to include quantiles use quantile(variable, collection of fractions between 0 and 1, na.rm=T)
quantile(sdf$M1, c(.25, .33, .5, .67, .75), na.rm=T)

25% 33% 50% 67% 75% 
 22  23  24  25  26 

In [ ]:
# use any parameter function of your choice and sort columns to your needs ...
tapply(sdf$M1, sdf$N1, median, na.rm=T)[c(1,3,2,4)] # Note: Indices define columns, no comma required

Austria Germany  EU/EEA   Other 
     24      24      23      23 

In [100]:
# ... or compile your individual selection, ...
Min = tapply(sdf$M1, sdf$N1, min, na.rm=T)
Q1 = tapply(sdf$M1, sdf$N1, quantile, .25, na.rm=T)
Median = tapply(sdf$M1, sdf$N1, median, na.rm=T)
Q3 = tapply(sdf$M1, sdf$N1, quantile, .75, na.rm=T)
Max = tapply(sdf$M1, sdf$N1, max, na.rm=T)
Mean = tapply(sdf$M1, sdf$N1, mean, na.rm=T)
SD = tapply(sdf$M1, sdf$N1, sd, na.rm=T)
n = as.numeric(table(sdf$N1, useNA="no"))
cbind(Min, Q1, Median, Q3, Max, Mean, SD,n)[c(1,3,2,4),]  # Note the position of the comma: Indices define rows!


        Min    Q1 Median    Q3 Max     Mean       SD   n
Austria  17 23.00     24 26.00  34 24.46919 2.651639 211
Germany  19 21.25     24 25.00  37 23.90000 3.319464  70
EU/EEA   19 22.00     23 24.00  37 23.56140 3.407018  58
Other    21 23.00     23 29.25  38 26.32143 5.171529  28

In [103]:
# ... formatted to your taste
rbind(Min, Q1, Median, Q3, Max, Mean, SD, n)[,c(1,3,2,4)]  # Note the position of the comma: Indices define columns!


          Austria   Germany    EU/EEA     Other
Min     17.000000 19.000000 19.000000 21.000000
Q1      23.000000 21.250000 22.000000 23.000000
Median  24.000000 24.000000 23.000000 23.000000
Q3      26.000000 25.000000 24.000000 29.250000
Max     34.000000 37.000000 37.000000 38.000000
Mean    24.469194 23.900000 23.561404 26.321429
SD       2.651639  3.319464  3.407018  5.171529
n      211.000000 70.000000 58.000000 28.000000

## Long scales as independent Variables (X)

When examining a metric independent variable in relation to a dependent variable with few response categories, grouping the metric variable into meaningful categories makes the results easier to present and interpret. In this example, the metric variable is divided into three approximately equal-sized groups using the 33rd and 67th percentiles as cut-off points. Use a different classification scheme if it is more appropriate.


In [104]:
# compute and check class bounds
terciles = quantile(sdf$M1, c(.33, .67), na.rm=T)
terciles

33% 67% 
 23  25 

In [105]:
# apply class bounds for recoding M1 into a new variable M1_3 (read: "M1 recoded into 3 classes")
sdf$M1_3 = NA
sdf$M1_3[sdf$M1 <= terciles[1]] = "low"
sdf$M1_3[sdf$M1 > terciles[1] & sdf$M1 <= terciles[2]] = "medium"
sdf$M1_3[sdf$M1 > terciles[2]] = "high"
table(sdf$M1_3)


  high    low medium 
   105    164    111 

Now continue with M1_3 to build a crosstable with your dependent variable of choice 